# 01 — Base model on Spider-dev (execution accuracy)

Day 1 goal: one number — base Qwen3-1.7B execution accuracy on Spider-dev (1,034) — plus tokens/latency per query.

Runtime → Change runtime type → **L4** (or A100 if your plan offers it).

In [ ]:
!nvidia-smi -L

In [ ]:
# Clone your repo (replace URL), or upload the src/ folder.
!git clone https://github.com/Waveorwaves/text2sql-minimal.git
%cd text2sql-minimal
!bash setup.sh

## Step 1 — 20 examples, sanity check
Look at a few raw outputs before trusting the metric. Common failures: model ignores the fence, emits explanation, uses wrong dialect.

In [ ]:
!python src/eval_local.py --model Qwen/Qwen3-1.7B --spider data/spider_data --out results/smoke --limit 20 --batch-size 8

In [ ]:
import json
for i, l in enumerate(open('results/smoke.predictions.jsonl')):
    d = json.loads(l)
    if i < 5 or not d['correct']:
        print('Q:', d['question']); print('GOLD:', d['gold_sql']); print('PRED:', d['pred_sql']); print('OK:', d['correct'], '| ERR:', d['error']); print('---')

## Step 2 — full dev set
~1,034 queries. On an L4 expect roughly 10–20 min at batch 16; A100 faster. If you hit OOM, lower --batch-size.

In [ ]:
!python src/eval_local.py --model Qwen/Qwen3-1.7B --spider data/spider_data --out results/base_qwen3-1.7b --batch-size 16

In [ ]:
import json; print(json.dumps(json.load(open('results/base_qwen3-1.7b.metrics.json')), indent=2))

## Step 3 — save results off Colab
Copy `results/*.metrics.json` (and predictions if you want) to Drive or commit them. Colab disk is wiped.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/text2sql-minimal && cp results/*.json results/*.jsonl /content/drive/MyDrive/text2sql-minimal/

## Optional — 0.6B for a size ablation
Same command with `--model Qwen/Qwen3-0.6B --out results/base_qwen3-0.6b`.

## Day 2 — frontier LLM on the same 1,034
Run locally or here. Look up the provider's per-token prices on the day and pass them in; they're written to metrics.json so the cost table is reproducible.

```
export OPENAI_API_KEY=...
python src/eval_api.py --model <frontier-model> --out results/api_<model> --price-in <usd/1M in> --price-out <usd/1M out>
```